# Smart Receipt Classifier & Expense Analytics
This notebook details the data processing, analysis, and category prediction modeling for the **Smart Receipt Scanner & Expense Logger** application.

---

## ☁️ Cloud Architecture & Data Ingestion
In a live production cloud environment, receipts are processed as follows:
1. **Frontend Upload**: A user uploads a receipt image via the mobile web UI (hosted on Vercel or GitHub Pages).
2. **Serverless Extraction**: A Python serverless backend function (hosted on Vercel) invokes the **Gemini Vision API** to extract structured JSON data from the receipt image (e.g. date, merchant, total, items, suggested category).
3. **Database Storage**: The extracted transaction is saved securely to a cloud database (such as **Supabase** or **Firestore**).
4. **Google Sheets / BI API Integration**: In enterprise or simple personal accounting scenarios, the backend can also automatically sync transaction rows to the **Google Sheets API** or a data warehouse for easy visibility.

For this analysis, we ingest the receipt history from `dataset_sample.csv`, which simulates a sync of transaction logs from our cloud database.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Set visual options
pd.set_option('display.max_columns', None)

## Step 1: Ingesting the Dataset
We load the receipts dataset and inspect the first few rows to understand the feature structure.

In [ ]:
# Ingest local receipt log dataset
df = pd.read_csv('dataset_sample.csv')
print(f"Total Receipt Records: {df.shape[0]}")
df.head()

## Step 2: Data Preprocessing & Cleaning
Before feeding features into a machine learning model, we clean the dataset:
- Parse `Total` amounts to numerical float values.
- Parse `Date` fields into Pandas datetime objects.
- Handle any missing/null values (imputing reasonable defaults).
- Check data types and structure.

In [ ]:
# Clean and prepare data
# ponytail: Simple cast and handle missing values inline
df['Total'] = pd.to_numeric(df['Total'], errors='coerce')
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Impute missing values if any exist
df['Total'] = df['Total'].fillna(0.0)
df['Merchant'] = df['Merchant'].fillna('Unknown')
df['Category'] = df['Category'].fillna('Others')

df.info()

## Step 3: Dataset Splitting (80% Train / 20% Test)
We split our dataset into a **Training Set (80%)** to build the classifier and a **Testing Set (20%)** to evaluate its out-of-sample performance.
We use stratification based on the target column `Category` to ensure class representation is proportional in both splits.

In [ ]:
# Select features (Merchant text and Total amount) and target category
X = df[['Merchant', 'Total']]
y = df['Category']

# Perform 80% train / 20% test split, stratified by category
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} receipts")
print(f"Testing set size: {X_test.shape[0]} receipts")

## Step 4: Model Design & Training
We build an ML classification pipeline:
1. **Feature Engineering / Column Transformation**:
   - Apply a **TF-IDF Vectorizer** to the `Merchant` names to capture textual keywords.
   - Apply a **StandardScaler** to the `Total` amount to scale numerical variance.
2. **Classifier**:
   - Train a simple, robust **Logistic Regression** classifier (which supports continuous features and text representation seamlessly).

In [ ]:
# Create preprocessing pipeline for mixed text and numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('merchant_tfidf', TfidfVectorizer(token_pattern=r'(?u)\b\w+\b'), 'Merchant'),
        ('total_std', StandardScaler(), ['Total'])
    ]
)

# Combine preprocessing and classifier into a single executable pipeline
# ponytail: Logistic regression handles combined tf-idf and continuous scaled Total values robustly
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Fit model on training data
pipeline.fit(X_train, y_train)
print("Pipeline model trained successfully!")

## Step 5: Evaluation on the Test Set
We predict categories for the test set and evaluate performance using accuracy, precision, recall, and a detailed classification report.

In [ ]:
# Predict categories on unseen test set
y_pred = pipeline.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

# Detailed classification metrics
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

## Step 6: Data Visualizations (Sunset Orange Aesthetic)
We plot custom-styled graphics to visualize:
1. **Spend breakdown per category**: Total expenditures per budget category.
2. **Spend trend over time**: Monthly spending trends.
3. **Model Confusion Matrix**: Correct versus incorrect category predictions on the test set.

We customize matplotlib to use a clean, minimalist **Sunset Orange** palette.

In [ ]:
# Set custom plot theme (Sunset Orange aesthetic)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = '#EAEAEA'
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#FAFAFA'
plt.rcParams['grid.linestyle'] = '-'
plt.rcParams['font.sans-serif'] = 'Segoe UI, Arial'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['text.color'] = '#2C3E50'
plt.rcParams['axes.labelcolor'] = '#2C3E50'
plt.rcParams['xtick.color'] = '#7F8C8D'
plt.rcParams['ytick.color'] = '#7F8C8D'

# Define Sunset Orange palette shades (Dark orange to light peach)
sunset_palette = ['#D35400', '#E67E22', '#F39C12', '#F5B041', '#EDBB99']

In [ ]:
# Chart 1: Spend Breakdown per Category
category_spend = df.groupby('Category')['Total'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(category_spend.index, category_spend.values, color=sunset_palette[:len(category_spend)], width=0.55)

# Style tweaks
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#EAEAEA')
ax.spines['bottom'].set_color('#EAEAEA')

# Annotate bars with values
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'${height:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4),  # 4 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9.5, color='#2C3E50', fontweight='semibold')

plt.title("Total Spend Breakdown by Category", fontsize=14, fontweight='bold', pad=22, color='#D35400')
plt.xlabel("Category", fontsize=11, labelpad=10)
plt.ylabel("Total Expenditure ($)", fontsize=11, labelpad=10)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Spend Trend Over Time
df_trend = df.copy()
df_trend['Month'] = df_trend['Date'].dt.to_period('M')
monthly_spend = df_trend.groupby('Month')['Total'].sum().sort_index()

months_str = [m.strftime('%b %Y') for m in monthly_spend.index]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(months_str, monthly_spend.values, color='#D35400', marker='o', linewidth=2.5, markersize=8)
ax.fill_between(months_str, monthly_spend.values, color='#E67E22', alpha=0.15)

# Style tweaks
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#EAEAEA')
ax.spines['bottom'].set_color('#EAEAEA')

# Annotate points
for i, val in enumerate(monthly_spend.values):
    ax.annotate(f'${val:.2f}',
                xy=(i, val),
                xytext=(0, 10),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9.5, color='#2C3E50', fontweight='semibold')

plt.title("Monthly Spending Trend (2026)", fontsize=14, fontweight='bold', pad=22, color='#D35400')
plt.xlabel("Month", fontsize=11, labelpad=10)
plt.ylabel("Total Expenditure ($)", fontsize=11, labelpad=10)
ax.set_ylim(0, max(monthly_spend.values) * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: Confusion Matrix Plot
fig, ax = plt.subplots(figsize=(6, 6))
cm = confusion_matrix(y_test, y_pred, labels=pipeline.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipeline.classes_)

# Plot heatmap using standard Oranges colormap
disp.plot(ax=ax, cmap=plt.cm.Oranges, colorbar=False)
ax.grid(False)  # Turn off grid inside confusion matrix

plt.title("Model Classification Confusion Matrix", fontsize=13, fontweight='bold', pad=22, color='#D35400')
plt.xlabel("Predicted Label", fontsize=11, labelpad=10)
plt.ylabel("True Label", fontsize=11, labelpad=10)
plt.tight_layout()
plt.show()